# "THE PRICE IS RIGHT" — Week 8, Day 2

## Week 8 Order of Play

Day 1: Modal.com and SpecialistAgent  
Day 2: RAG, FrontierAgent, Ensemble Agent  
Day 3: ScannerAgent, MessengerAgent  
Day 4: AutonomousPlannerAgent and DealAgentFramework  
Day 5: The Price Is Right Finale

## RAG (Retrieval Augmented Generation) based on a dataset of 800,000 scraped Amazon products

#### For our 2nd agent, we will be asking OpenAI to estimate the price of one of our deals - and we will give it a hand.

We discovered that LLMs are really good at this, out of the box.

And we discovered that we can beat a frontier LLM by fine-tuning an open-source LLM.

Now we are going to try **inference time** techniques instead of training -- by using RAG!

In [1]:
# imports

import os
import logging
from pathlib import Path
from dotenv import find_dotenv, load_dotenv
from huggingface_hub import login
import numpy as np
import re
from sentence_transformers import SentenceTransformer
import chromadb
from sklearn.manifold import TSNE
import plotly.graph_objects as go
from litellm import completion
from tqdm.auto import tqdm
from agents.evaluator import evaluate
from agents.items import Item

/Users/marcolerma/GitHub/applied-llm-engineering/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


In [2]:
# Environment and paths

dotenv_path = find_dotenv(usecwd=True)
if not dotenv_path:
    raise FileNotFoundError("Could not find the repository .env file")
load_dotenv(dotenv_path, override=True)

REPO_ROOT = Path(dotenv_path).parent
WEEK8_DIR = REPO_ROOT / "lectures" / "week-eight"
DB = str(WEEK8_DIR / "products_vectorstore")
OPENAI_MODEL = os.getenv("PRICER_PREPROCESSOR_MODEL", "openai/gpt-5-nano")

In [3]:
# Optional Hugging Face login. The course datasets are public, so a token is not required.
hf_token = os.getenv("HF_TOKEN")
if hf_token:
    login(token=hf_token, add_to_git_credential=False)
else:
    print("HF_TOKEN is not set; continuing with public Hugging Face access.")

HF_TOKEN is not set; continuing with public Hugging Face access.


In [4]:
# Start with the smaller dataset; set False only when you are ready for the full build.
LITE_MODE = True

In [5]:
username = "marcolerma"
dataset = f"{username}/items_lite" if LITE_MODE else f"{username}/items_full"

train, val, test = Item.from_hub(dataset)

print(f"Loaded {len(train):,} training items, {len(val):,} validation items, {len(test):,} test items")

Loaded 20,000 training items, 1,000 validation items, 1,000 test items


# Now create a Chroma Datastore

Now we will use the free, open-source Vector database Chroma.  
We will create a Chroma datastore with 400,000 products from our training dataset.

In [6]:
client = chromadb.PersistentClient(path=DB)

# Introducing the SentenceTransformer Encoding LLM

The all-MiniLM is a very useful model from HuggingFace that maps sentences & paragraphs to 384 dimensional vectors and is ideal for tasks like semantic search.

https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2

It can run pretty quickly locally.

As an alternative, OpenAI provides a closed-source Embeddings model. Benefits compared to OpenAI embeddings:
1. It's free and fast!
3. We can run it locally, so the data never leaves our box - might be useful if you're building a personal RAG

In [7]:
encoder = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')

In [8]:
# Pass in a list of texts, get back a numpy array of vectors

vector = encoder.encode(["A proficient AI engineer who has almost reached the finale of AI Engineering Core Track!"])[0]
print(vector.shape)
vector

(384,)


array([-5.68234101e-02, -6.70465305e-02,  4.41130325e-02,  5.98601578e-03,
       -2.28948798e-02, -2.95300186e-02,  5.56369536e-02,  3.42665762e-02,
       -1.08529359e-01, -3.81690785e-02, -7.43872151e-02, -1.03664227e-01,
        1.69147998e-02,  1.33921846e-03, -6.86191544e-02,  8.99353623e-02,
       -1.45186651e-02, -2.43885871e-02,  4.21849173e-03, -9.62991863e-02,
       -2.51799505e-02,  4.60676588e-02,  4.95347101e-03, -3.88679393e-02,
        1.07716850e-03,  6.82337210e-02, -1.13859568e-02, -5.83416522e-02,
       -1.03801060e-02, -1.74952932e-02, -1.86478850e-02,  4.07059770e-03,
        1.59438029e-02,  6.49721995e-02,  3.71175818e-02,  2.78225653e-02,
       -4.41945791e-02, -2.34372541e-02,  9.71035138e-02, -5.06139211e-02,
       -1.93864387e-02, -3.83472517e-02,  4.76066582e-02, -3.36106606e-02,
        5.08287251e-02,  3.57935429e-02,  2.91816122e-03, -1.06529169e-01,
        4.07212004e-02, -5.85448055e-04, -1.05607450e-01, -1.03584342e-01,
        3.71124111e-02, -

## Populate the Chroma database

The full dataset can take 30 minutes or longer and uses substantial disk and memory. This notebook defaults to `LITE_MODE = True`; switch it off only when you are ready for the full build.

In [9]:
# Some dataset rows do not have a summary, so choose the best available text.
def item_text(item):
    for value in (item.summary, item.full, item.title, item.prompt):
        if isinstance(value, str) and value.strip():
            return value.strip()
    raise ValueError(f"Item {item.id!r} has no usable text")

# Use separate collections for lite/full data and make ingestion safe to rerun.
collection_name = "products_lite" if LITE_MODE else "products_full"
collection = client.get_or_create_collection(collection_name)

if collection.count() < len(train):
    for i in tqdm(range(0, len(train), 1000), desc="Indexing products"):
        batch = train[i: i+1000]
        documents = [item_text(item) for item in batch]
        embeddings = encoder.encode(documents).astype(float).tolist()
        metadatas = [{"category": item.category, "price": item.price} for item in batch]
        ids = [f"doc_{j}" for j in range(i, i + len(batch))]
        # upsert makes this cell recover cleanly after an interrupted run.
        collection.upsert(ids=ids, documents=documents, embeddings=embeddings, metadatas=metadatas)
else:
    print(f"Collection already contains {collection.count():,} products.")

print(f"Using Chroma collection '{collection_name}' with {collection.count():,} products.")

Indexing products:   0%|          | 0/20 [00:00<?, ?it/s]

Using Chroma collection 'products_lite' with 20,000 products.


# Let's visualize the vectorized data

In [10]:
# It is very fun turning this up to 800_000 and seeing the full dataset visualized,
# but it almost crashes my box every time so do that at your own risk!! 10_000 is safe!

MAXIMUM_DATAPOINTS = 10_000

In [11]:
CATEGORIES = ['Appliances', 'Automotive', 'Cell_Phones_and_Accessories', 'Electronics','Musical_Instruments', 'Office_Products', 'Tools_and_Home_Improvement', 'Toys_and_Games']
COLORS = ['cyan', 'blue', 'brown', 'orange', 'yellow', 'green' , 'purple', 'red']

In [12]:
# Prework
result = collection.get(include=['embeddings', 'documents', 'metadatas'], limit=MAXIMUM_DATAPOINTS)
vectors = np.array(result['embeddings'])
documents = result['documents']
categories = [metadata['category'] for metadata in result['metadatas']]
colors = [COLORS[CATEGORIES.index(c)] for c in categories]

n_samples = len(vectors)
if n_samples < 4:
    raise RuntimeError(
        f"The collection contains only {n_samples} products. Rerun the ingestion cell above before creating the t-SNE charts."
    )
TSNE_PERPLEXITY = min(30, n_samples - 1)
TSNE_OPTIONS = {"perplexity": TSNE_PERPLEXITY, "random_state": 42}
print(f"Visualizing {n_samples:,} products with t-SNE perplexity={TSNE_PERPLEXITY}.")

Visualizing 10,000 products with t-SNE perplexity=30.


In [13]:
# Let's try a 2D chart
# TSNE stands for t-distributed Stochastic Neighbor Embedding - it's a common technique for reducing dimensionality of data

tsne = TSNE(n_components=2, **TSNE_OPTIONS)
reduced_vectors = tsne.fit_transform(vectors)

In [14]:
# Create the 2D scatter plot
fig = go.Figure(data=[go.Scatter(
    x=reduced_vectors[:, 0],
    y=reduced_vectors[:, 1],
    mode='markers',
    marker=dict(size=4, color=colors, opacity=0.7),
    text=[f"Category: {c}<br>Text: {d[:50]}..." for c, d in zip(categories, documents)],
    hoverinfo='text'
)])

fig.update_layout(
    title='2D Chroma Vectorstore Visualization',
    scene=dict(xaxis_title='x', yaxis_title='y'),
    width=1200,
    height=800,
    margin=dict(r=20, b=10, l=10, t=40)
)

fig.show()

In [15]:
# Let's try 3D!

tsne = TSNE(n_components=3, **TSNE_OPTIONS)
reduced_vectors = tsne.fit_transform(vectors)

In [16]:
# Create the 3D scatter plot
fig = go.Figure(data=[go.Scatter3d(
    x=reduced_vectors[:, 0],
    y=reduced_vectors[:, 1],
    z=reduced_vectors[:, 2],
    mode='markers',
    marker=dict(size=2, color=colors, opacity=0.7),
    text=[f"Category: {c}<br>Text: {d[:50]}..." for c, d in zip(categories, documents)],
    hoverinfo='text'
)])

fig.update_layout(
    title='3D Chroma Vector Store Visualization',
    scene=dict(xaxis_title='x', yaxis_title='y', zaxis_title='z'),
    width=1200,
    height=800,
    margin=dict(r=20, b=10, l=10, t=40)
)

fig.show()

In [17]:
test[0]

<Old Blood Noise Excess V2 Distortion Chorus/Delay Pedal = $219.0>

In [18]:
def vector(item):
    return encoder.encode(item_text(item))

In [19]:
def find_similars(item):
    vec = vector(item)
    results = collection.query(query_embeddings=vec.astype(float).tolist(), n_results=5)
    documents = results['documents'][0][:]
    prices = [m['price'] for m in results['metadatas'][0][:]]
    return documents, prices

In [20]:
find_similars(test[0])

(['Fender The Bends Compressor Pedal\n[\'Dynamics are a crucial part of playing music. When controlled they bring music to life, but uneven or wild playing can ruin a performance. We put our expertise to work creating the bends compressor, a studio-quality stomp box that tames wild volume spikes without altering your tone. Drive and recovery controls let you dial in the perfect amount of compression to complement your playing style and extend sustain, while the blend control lets you mix in the dry signal to maintain your natural pick attack. The amp jewel LED changes color from white to pink along with your playing to help show when the compression circuit is engaged and how long your signal is affect.\']\n[\'Dual internal audio paths for low noise\', \'High-current symmetrical control path for fast response time\', \'Led backlit knobs\', \'Fender amp Jewel LED\', \'Magnetically latched hinged 9V battery door\']\n{"Item Weight": "454 Grams", "Product Dimensions": "5.45 x 4.85 x 3.25 i

In [21]:
# We need to give some context to GPT-5 nano by selecting 5 products with similar descriptions

def make_context(similars, prices):
    message = "For context, here are some other items that might be similar to the item you need to estimate.\n\n"
    for similar, price in zip(similars, prices):
        message += f"Potentially related product:\n{similar}\nPrice is ${price:.2f}\n\n"
    return message

In [22]:
documents, prices = find_similars(test[0])
print(make_context(documents, prices))

For context, here are some other items that might be similar to the item you need to estimate.

Potentially related product:
Fender The Bends Compressor Pedal
['Dynamics are a crucial part of playing music. When controlled they bring music to life, but uneven or wild playing can ruin a performance. We put our expertise to work creating the bends compressor, a studio-quality stomp box that tames wild volume spikes without altering your tone. Drive and recovery controls let you dial in the perfect amount of compression to complement your playing style and extend sustain, while the blend control lets you mix in the dry signal to maintain your natural pick attack. The amp jewel LED changes color from white to pink along with your playing to help show when the compression circuit is engaged and how long your signal is affect.']
['Dual internal audio paths for low noise', 'High-current symmetrical control path for fast response time', 'Led backlit knobs', 'Fender amp Jewel LED', 'Magneticall

In [23]:
def messages_for(item, similars, prices):
    message = f"Estimate the price of this product. Respond with the price, no explanation\n\n{item_text(item)}\n\n"
    message += make_context(similars, prices)
    return [{"role": "user", "content": message}]

In [24]:
documents, prices = find_similars(test[0])
print(messages_for(test[0], documents, prices)[0]['content'])

Estimate the price of this product. Respond with the price, no explanation

Old Blood Noise Excess V2 Distortion Chorus/Delay Pedal
['Expanding on the Excess, Old Blood Noise continues their journey to revive and expand sought-out pedal sounds of the 80s. With delay, chorus, and harmonized fifths available along with distortion in series or parallel configurations, Excess V2 unveils more sounds and even more control.']
['Three modes of modulation (Delay, Chorus, and Fifth) each with Time, Depth, and Volume controls', 'Distortion section with control over Gain, Tone, and Volume to access light overdrive through fuzzy distortion', 'Order switching to run Distortion into Modulation, Modulation into Distortion, or separate parallel paths', 'Soft touch switching and true relay bypass for each section', 'Expression jack to externally control Rate and Depth', 'Internal trimpot to set wet-dry mix of Modulation section', 'Requires at least 125mA 9VDC center negative power']
{"Item Weight": "2 P

In [25]:
# RAG price estimator using the OpenAI model configured in .env

def gpt_5_nano_rag(item):
    documents, prices = find_similars(item)
    response = completion(model=OPENAI_MODEL, messages=messages_for(item, documents, prices))
    return response.choices[0].message.content

In [26]:
# How much does our favorite distortion pedal cost?

test[0].price

219.0

In [27]:
# Estimate one product before running the larger evaluation.
gpt_5_nano_rag(test[0])

'$199.99'

In [28]:
# This makes many paid API calls; start with a small sample.
evaluate(gpt_5_nano_rag, test, size=10, workers=2)

  0%|          | 0/10 [00:00<?, ?it/s]

$19 $74 $25 $45 $0 $0 $94 $55 $3 $181 

In [29]:
import modal
Pricer = modal.Cls.from_name("pricer-service", "Pricer")
pricer = Pricer()

In [30]:
def specialist(item):
    return pricer.price.remote(item_text(item))


In [31]:
def get_price(reply):
    reply = reply.replace("$", "").replace(",", "")
    match = re.search(r"[-+]?\d*\.\d+|\d+", reply)
    return float(match.group()) if match else 0

## Download the neural-network weights from Week 6

Download `deep_neural_network.pth` into the `lectures/week-eight/` directory from the course folder:

https://drive.google.com/drive/folders/1uq5C9edPIZ1973dArZiEO-VE13F7m8MK?usp=drive_link

The RAG and Frontier Agent sections work without this file. The neural-network and ensemble sections require it.

In [35]:
from agents.deep_neural_network import DeepNeuralNetworkInference

weights_path = WEEK8_DIR / "deep_neural_network.pth"
if not weights_path.exists():
    raise FileNotFoundError(f"Download the Week 6 weights to {weights_path}")

runner = DeepNeuralNetworkInference()
runner.setup()
runner.load(weights_path)

def deep_neural_network(item):
    return runner.inference(item_text(item))

In [36]:
def ensemble(item):
    price1 = get_price(gpt_5_nano_rag(item))
    price2 = specialist(item)
    price3 = deep_neural_network(item)
    return price1 * 0.8 + price2 * 0.1 + price3 * 0.1


In [37]:
evaluate(ensemble, test)

  0%|          | 0/200 [00:00<?, ?it/s]

$75 $57 $21 $7 $4 $19 $93 $42 $7 $128 $293 $6 $19 $16 $12 $2 $23 $13 $9 $94 $23 $157 $54 $21 $181 $227 $80 $2 $108 $58 $27 $18 $46 $32 $8 $351 $18 $21 $69 $4 $82 $29 $20 $59 $71 $7 $8 $0 $74 $16 $15 $105 $134 $8 $98 $42 $5 $91 $7 $1 $101 $23 $36 $113 $240 $30 $76 $264 $26 $68 $18 $1 $123 $14 $14 $16 $31 $21 $2 $2 $26 $39 $34 $43 $9 $52 $30 $92 $20 $6 $6 $1 $3 $1 $4 $92 $1 $12 $98 $60 $49 $42 $0 $3 $1 $102 $1 $344 $26 $2 $21 $3 $2 $38 $54 $47 $13 $7 $53 $27 $14 $55 $48 $33 $27 $17 $16 $58 $38 $49 $3 $40 $2 $6 $14 $1 $91 $8 $18 $27 $17 $78 $28 $5 $8 $27 $8 $112 $31 $17 $3 $47 $23 $42 $2 $6 $94 $24 $87 $9 $25 $17 $27 $6 $327 $15 $65 $18 $4 $16 $1 $2 $75 $2 $23 $23 $12 $49 $18 $28 $96 $10 $146 $56 $8 $18 $48 $39 $30 $8 $7 $25 $15 $5 $2 $61 $18 $29 $14 $2 

In [38]:
root = logging.getLogger()
root.setLevel(logging.INFO)

In [39]:
from agents.frontier_agent import FrontierAgent

agent = FrontierAgent(collection)
agent.price("Quadcast HyperX condenser mic, connects via usb-c to your computer for crystal clear audio")

INFO:root:[Frontier Agent] Initializing Frontier Agent
INFO:root:[Frontier Agent] Frontier Agent is setting up with OpenAI
INFO:sentence_transformers.SentenceTransformer:Use pytorch device_name: mps
INFO:sentence_transformers.SentenceTransformer:Load pretrained SentenceTransformer: sentence-transformers/all-MiniLM-L6-v2
INFO:root:[Frontier Agent] Frontier Agent is ready
INFO:root:[Frontier Agent] Frontier Agent is performing a RAG search of the Chroma datastore to find 5 similar products


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

INFO:root:[Frontier Agent] Frontier Agent has found similar products
INFO:root:[Frontier Agent] Frontier Agent is about to call gpt-5-nano with context including 5 similar products
INFO:root:[Frontier Agent] Frontier Agent completed - predicting $129.99


129.99

In [40]:
agent.price("Shure MV7+ professional podcaster microphone with usb-c and XLR outputs")

INFO:root:[Frontier Agent] Frontier Agent is performing a RAG search of the Chroma datastore to find 5 similar products


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

INFO:root:[Frontier Agent] Frontier Agent has found similar products
INFO:root:[Frontier Agent] Frontier Agent is about to call gpt-5-nano with context including 5 similar products
INFO:root:[Frontier Agent] Frontier Agent completed - predicting $299.00


299.0

In [41]:
from agents.neural_network_agent import NeuralNetworkAgent
agent = NeuralNetworkAgent()


INFO:root:[Neural Network Agent] Neural Network Agent is initializing
INFO:root:Neural Network is using mps
INFO:root:[Neural Network Agent] Neural Network Agent is ready and weights are loaded


In [42]:
agent.price("Shure MV7+ professional podcaster microphone with usb-c and XLR outputs")

INFO:root:[Neural Network Agent] Neural Network Agent is starting a prediction
INFO:root:[Neural Network Agent] Neural Network Agent completed - predicting $162.39


162.3922576904297

In [43]:
from agents.ensemble_agent import EnsembleAgent
agent = EnsembleAgent(collection)

INFO:root:[Ensemble Agent] Initializing Ensemble Agent
INFO:root:[Specialist Agent] Specialist Agent is initializing - connecting to modal
INFO:root:[Frontier Agent] Initializing Frontier Agent
INFO:root:[Frontier Agent] Frontier Agent is setting up with OpenAI
INFO:sentence_transformers.SentenceTransformer:Use pytorch device_name: mps
INFO:sentence_transformers.SentenceTransformer:Load pretrained SentenceTransformer: sentence-transformers/all-MiniLM-L6-v2
INFO:root:[Frontier Agent] Frontier Agent is ready
INFO:root:[Neural Network Agent] Neural Network Agent is initializing
INFO:root:Neural Network is using mps
INFO:root:[Neural Network Agent] Neural Network Agent is ready and weights are loaded
INFO:root:[Ensemble Agent] Ensemble Agent is ready


In [44]:
agent.price("Shure MV7+ professional podcaster microphone with usb-c and XLR outputs")

INFO:root:[Ensemble Agent] Running Ensemble Agent - preprocessing text
14:47:21 - LiteLLM:INFO: utils.py:4011 - 
LiteLLM completion() model= gpt-5-nano; provider = openai
INFO:LiteLLM:
LiteLLM completion() model= gpt-5-nano; provider = openai
14:47:29 - LiteLLM:INFO: utils.py:1653 - Wrapper: Completed Call, calling success_handler
INFO:LiteLLM:Wrapper: Completed Call, calling success_handler
INFO:root:[Ensemble Agent] Pre-processed text using openai/gpt-5-nano
INFO:root:[Specialist Agent] Specialist Agent is calling remote fine-tuned model
INFO:root:[Specialist Agent] Specialist Agent completed - predicting $299.00
INFO:root:[Frontier Agent] Frontier Agent is performing a RAG search of the Chroma datastore to find 5 similar products


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

INFO:root:[Frontier Agent] Frontier Agent has found similar products
INFO:root:[Frontier Agent] Frontier Agent is about to call gpt-5-nano with context including 5 similar products
INFO:root:[Frontier Agent] Frontier Agent completed - predicting $299.99
INFO:root:[Neural Network Agent] Neural Network Agent is starting a prediction
INFO:root:[Neural Network Agent] Neural Network Agent completed - predicting $112.37
INFO:root:[Ensemble Agent] Ensemble Agent complete - returning $281.13


281.12919787597656